In [0]:
# Quick Spark test
spark.range(10).show()

In [0]:
# Install YouTube API client (classic way in Databricks)
%pip install google-api-python-client --quiet
dbutils.library.restartPython()  # Restart Python to load library

In [0]:
# -------------------------------
# Step 1: Set your variables
# -------------------------------

# Automatically get workspace URL and token from notebook context
WORKSPACE_URL = "https://" + spark.conf.get("spark.databricks.workspaceUrl")
PAT = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
SCOPE_NAME = "youtube-scope"
SECRET_KEY = "youtube-api-key"
SECRET_VALUE = ""


# -------------------------------
# Step 2: Import libraries
# -------------------------------
import requests
import json

# Headers for authentication
HEADERS = {
    "Authorization": f"Bearer {PAT}",
    "Content-Type": "application/json"
}

# -------------------------------
# Step 3: Create Secret Scope
# -------------------------------
create_scope_url = f"{WORKSPACE_URL}/api/2.0/secrets/scopes/create"

payload_scope = {
    "scope": SCOPE_NAME,
    "scope_backend_type": "DATABRICKS"  # Simple Databricks-managed scope
}

# Make request
response_scope = requests.post(create_scope_url, headers=HEADERS, data=json.dumps(payload_scope))

if response_scope.status_code == 200:
    print(f"✅ Secret scope '{SCOPE_NAME}' created successfully!")
elif "ResourceAlreadyExistsException" in response_scope.text:
    print(f"⚠️ Secret scope '{SCOPE_NAME}' already exists, skipping creation.")
else:
    print("❌ Error creating secret scope:", response_scope.text)

# -------------------------------
# Step 4: Put secret (API key)
# -------------------------------
put_secret_url = f"{WORKSPACE_URL}/api/2.0/secrets/put"

payload_secret = {
    "scope": SCOPE_NAME,
    "key": SECRET_KEY,
    "string_value": SECRET_VALUE
}

response_secret = requests.post(put_secret_url, headers=HEADERS, data=json.dumps(payload_secret))

if response_secret.status_code == 200:
    print(f"✅ Secret '{SECRET_KEY}' stored successfully in scope '{SCOPE_NAME}'")
else:
    print("❌ Error storing secret:", response_secret.text)

# -------------------------------
# Step 5: Test retrieving secret in notebook
# -------------------------------
# This is the safe way to use your secret in ETL
api_key = dbutils.secrets.get(scope=SCOPE_NAME, key=SECRET_KEY)
print(f"Secret retrieved successfully: Length = {len(api_key)} (value hidden)")

In [0]:
# Get API key from secret scope
api_key = dbutils.secrets.get(scope="youtube-scope", key="youtube-api-key")

!pip install --quiet google-api-python-client
from googleapiclient.discovery import build


youtube = build("youtube", "v3", developerKey=api_key)
print("YouTube API client initialized successfully!")

In [0]:
spark.sql("SHOW TABLES IN workspace_yt_mar.gold").show()

In [0]:
CATALOG = "workspace_yt_mar"
SCHEMAS = ["bronze", "silver", "gold"]
for schema in SCHEMAS:
     spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{schema}")
     spark.sql(f"USE {CATALOG}.{schema}")

print("Schemas created successfully")

In [0]:
spark.sql("TRUNCATE TABLE workspace_yt_mar.bronze.channels_raw")

In [0]:
spark.sql("DROP TABLE IF EXISTS workspace_yt_mar.gold.video_type_summary")